**Model Architecture**

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# BAYESIAN LINEAR LAYER
# =========================================================
class BayesianLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, prior_std: float = 1.0):
        super().__init__()

        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Mean parameters
        self.weight_mean = nn.Parameter(torch.randn(out_features, in_features) * 0.1)
        self.bias_mean = nn.Parameter(torch.randn(out_features) * 0.1)

        # Log variance (log σ²)
        self.weight_logvar = nn.Parameter(torch.randn(out_features, in_features) * 0.1 - 5)
        self.bias_logvar = nn.Parameter(torch.randn(out_features) * 0.1 - 5)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        if sample:
            weight_std = torch.exp(0.5 * self.weight_logvar)
            bias_std = torch.exp(0.5 * self.bias_logvar)

            weight = self.weight_mean + weight_std * torch.randn_like(weight_std)
            bias = self.bias_mean + bias_std * torch.randn_like(bias_std)
        else:
            weight = self.weight_mean
            bias = self.bias_mean

        return F.linear(x, weight, bias)

    def get_kl_divergence(self) -> torch.Tensor:
        """
        KL divergence between:
        q(w|θ) = N(μ, σ²)
        p(w)   = N(0, prior_std²)
        """
        weight_var = torch.exp(self.weight_logvar)
        bias_var = torch.exp(self.bias_logvar)

        prior_var = torch.tensor(self.prior_std ** 2, device=weight_var.device)

        kl_weight = 0.5 * torch.sum(
            (weight_var + self.weight_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.weight_logvar
        )

        kl_bias = 0.5 * torch.sum(
            (bias_var + self.bias_mean**2) / prior_var
            - 1.0
            + torch.log(prior_var)
            - self.bias_logvar
        )

        return kl_weight + kl_bias


# =========================================================
# BAYESIAN HEAD
# =========================================================
class BayesianDetectionHead(nn.Module):
    def __init__(self, input_channels: int = 512):
        super().__init__()

        self.fc1 = BayesianLinear(input_channels, 256)
        self.fc2 = BayesianLinear(256, 128)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        x = self.fc1(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x, sample)
        x = self.relu(x)
        x = self.dropout(x)

        return x

    def get_kl_divergence(self) -> torch.Tensor:
        return self.fc1.get_kl_divergence() + self.fc2.get_kl_divergence()


# =========================================================
# B-YOLO MODEL
# =========================================================
class B_YOLO(nn.Module):
    def __init__(self, backbone: nn.Module, num_classes: int = 80):
        super().__init__()

        self.backbone = backbone
        self.num_classes = num_classes

        # Freeze backbone properly ✔️
        for p in self.backbone.parameters():
            p.requires_grad = False

        self.bayesian_head = BayesianDetectionHead(512)

        # Task heads
        self.detection_head = nn.Linear(128, 4 + num_classes)
        self.segmentation_head = nn.Linear(128, 256)
        self.pose_head = nn.Linear(128, 17 * 2)
        self.oriented_head = nn.Linear(128, 5 + num_classes)
        self.classification_head = nn.Linear(128, num_classes)

        self.current_task = None

    def set_task(self, task: str):
        valid_tasks = ['detection', 'segmentation', 'pose', 'oriented', 'classification']
        if task not in valid_tasks:
            raise ValueError(f"Task must be one of {valid_tasks}")
        self.current_task = task

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        # Backbone
        with torch.no_grad():
            features = self.backbone(x)

        # Flatten
        if features.dim() == 4:
            features = F.adaptive_avg_pool2d(features, 1)
            features = features.view(features.size(0), -1)

        # Bayesian head
        x = self.bayesian_head(features, sample)

        # Task-specific output
        if self.current_task == 'detection':
            return self.detection_head(x)

        elif self.current_task == 'segmentation':
            return self.segmentation_head(x)

        elif self.current_task == 'pose':
            return self.pose_head(x)

        elif self.current_task == 'oriented':
            return self.oriented_head(x)

        elif self.current_task == 'classification':
            return self.classification_head(x)

        else:
            raise ValueError("Task not set")

    def predict_with_uncertainty(self, x: torch.Tensor, T: int = 10):
        """
        Monte Carlo inference
        """
        self.eval()

        preds = []
        with torch.no_grad():
            for _ in range(T):
                preds.append(self.forward(x, sample=True))

        preds = torch.stack(preds)  # (T, B, D)

        if self.current_task == 'classification':
            probs = torch.softmax(preds, dim=-1)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'detection':
            bbox = preds[..., :4]
            logits = preds[..., 4:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'oriented':
            bbox = preds[..., :4]
            angle = preds[..., 4:5]
            logits = preds[..., 5:]

            probs = torch.softmax(logits, dim=-1)

            return {
                'bbox_mean': bbox.mean(0),
                'bbox_uncertainty': bbox.var(0),
                'angle_mean': angle.mean(0),
                'angle_uncertainty': angle.var(0),
                'class_probs_mean': probs.mean(0),
                'class_uncertainty': probs.var(0),
                'confidence': probs.mean(0).max(dim=1)[0]
            }

        elif self.current_task == 'segmentation':
            probs = torch.sigmoid(preds)
            return {
                'mean': probs.mean(0),
                'uncertainty': probs.var(0)
            }

        elif self.current_task == 'pose':
            return {
                'mean': preds.mean(0),
                'uncertainty': preds.var(0)
            }

        else:
            raise ValueError("Unknown task")

    def get_kl_divergence(self):
        return self.bayesian_head.get_kl_divergence()

**Task-Specific Loss Functions**

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F


# =========================================================
# TASK-SPECIFIC LOSS
# =========================================================
class TaskSpecificLoss(nn.Module):
    def __init__(self, task: str, num_classes: int = 80):
        super().__init__()
        self.task = task
        self.num_classes = num_classes

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        kl_divergence: torch.Tensor = None,
        kl_weight: float = 0.01
    ) -> torch.Tensor:

        if self.task == 'detection':
            task_loss = self._detection_loss(predictions, targets)

        elif self.task == 'segmentation':
            task_loss = self._segmentation_loss(predictions, targets)

        elif self.task == 'pose':
            task_loss = self._pose_loss(predictions, targets)

        elif self.task == 'oriented':
            task_loss = self._oriented_loss(predictions, targets)

        elif self.task == 'classification':
            task_loss = self._classification_loss(predictions, targets)

        else:
            raise ValueError(f"Unknown task: {self.task}")

        if kl_divergence is not None:
            task_loss = task_loss + kl_weight * kl_divergence

        return task_loss

    # =====================================================
    # DETECTION
    # =====================================================
    def _detection_loss(self, predictions, targets):
        bbox_pred = predictions[:, :4]
        bbox_target = targets[:, :4]

        bbox_loss = F.smooth_l1_loss(bbox_pred, bbox_target)

        class_pred = predictions[:, 4:]

        if targets.dim() == 2:
            class_target = targets[:, 4:]
            if class_target.shape[1] == self.num_classes:
                class_target = class_target.argmax(dim=1)
            else:
                raise ValueError("Invalid class target shape")
        else:
            raise ValueError("Targets must be 2D")

        class_loss = F.cross_entropy(class_pred, class_target)

        return bbox_loss * 0.5 + class_loss

    # =====================================================
    # SEGMENTATION
    # =====================================================
    def _segmentation_loss(self, predictions, targets):
        bce = F.binary_cross_entropy_with_logits(predictions, targets)

        probs = torch.sigmoid(predictions)

        intersection = (probs * targets).sum(dim=1)
        union = probs.sum(dim=1) + targets.sum(dim=1)

        dice = 1 - ((2 * intersection + 1e-8) / (union + 1e-8))
        dice = dice.mean()

        return 0.5 * bce + 0.5 * dice

    # =====================================================
    # POSE
    # =====================================================
    def _pose_loss(self, predictions, targets):
        mse = F.mse_loss(predictions, targets)
        smooth = F.smooth_l1_loss(predictions, targets)
        return 0.7 * mse + 0.3 * smooth

    # =====================================================
    # ORIENTED DETECTION
    # =====================================================
    def _oriented_loss(self, predictions, targets):
        bbox_loss = F.smooth_l1_loss(predictions[:, :4], targets[:, :4])

        angle_pred = predictions[:, 4]
        angle_target = targets[:, 4]

        angle_diff = torch.atan2(
            torch.sin(angle_pred - angle_target),
            torch.cos(angle_pred - angle_target)
        )

        angle_loss = torch.mean(angle_diff ** 2)

        class_pred = predictions[:, 5:]

        class_target = targets[:, 5:]
        if class_target.shape[1] == self.num_classes:
            class_target = class_target.argmax(dim=1)

        class_loss = F.cross_entropy(class_pred, class_target)

        return 0.4 * bbox_loss + 0.3 * angle_loss + 0.3 * class_loss

    # =====================================================
    # CLASSIFICATION
    # =====================================================
    def _classification_loss(self, predictions, targets):
        if targets.dim() == 2:
            targets = targets.argmax(dim=1)

        return F.cross_entropy(predictions, targets)


# =========================================================
# EWC REGULARIZER
# =========================================================
class EWCRegularizer(nn.Module):
    def __init__(self, model: nn.Module, lambda_ewc: float = 0.4):
        super().__init__()
        self.model = model
        self.lambda_ewc = lambda_ewc

        self.fisher_information = {}
        self.previous_weights = {}

    def compute_fisher_information(
        self,
        dataloader,
        loss_fn,
        device
    ):
        self.model.eval()

        fisher = {
            name: torch.zeros_like(param, device=device)
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

        total_samples = 0

        for images, targets in dataloader:
            images = images.to(device)
            targets = targets.to(device)

            self.model.zero_grad()

            outputs = self.model(images, sample=False)
            loss = loss_fn(outputs, targets)

            loss.backward()

            for name, param in self.model.named_parameters():
                if param.grad is not None:
                    fisher[name] += (param.grad.detach() ** 2)

            total_samples += 1

        for name in fisher:
            fisher[name] /= max(total_samples, 1)
            fisher[name] += 1e-8  # stability

        self.fisher_information = fisher
        self._save_previous_weights()

    def _save_previous_weights(self):
        self.previous_weights = {
            name: param.detach().clone()
            for name, param in self.model.named_parameters()
            if param.requires_grad
        }

    def get_ewc_loss(self):
        if not self.fisher_information:
            return torch.tensor(0.0, device=next(self.model.parameters()).device)

        loss = torch.zeros(1, device=next(self.model.parameters()).device)

        for name, param in self.model.named_parameters():
            if name in self.fisher_information:
                fisher = self.fisher_information[name]
                prev = self.previous_weights[name]

                loss += torch.sum(fisher * (param - prev) ** 2)

        return (self.lambda_ewc / 2.0) * loss

**Training Loop with Best Model Saving**

In [3]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import os
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict


class B_YOLOTrainer:
    def __init__(
        self,
        model,
        tasks,
        device=None,
        checkpoint_dir='./checkpoints'
    ):
        self.model = model
        self.tasks = tasks
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')

        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.ewc_regularizer = EWCRegularizer(model)

        self.best_metrics = defaultdict(lambda: {'score': 0.0, 'model_path': None})
        self.training_history = defaultdict(list)

    # =====================================================
    # MAIN TRAINING LOOP
    # =====================================================
    def train_sequential_tasks(
        self,
        task_dataloaders,
        num_epochs_per_task=50,
        learning_rate=1e-3,
        kl_weight=0.01,
        lambda_ewc=0.4
    ):
        self.model.to(self.device)

        for task_idx, task in enumerate(self.tasks):

            print(f"\n{'='*60}")
            print(f"Task {task_idx+1}/{len(self.tasks)}: {task.upper()}")
            print(f"{'='*60}")

            self.model.set_task(task)

            loss_fn = TaskSpecificLoss(task, self.model.num_classes)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=learning_rate
            )

            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5
            )

            train_loader = task_dataloaders[task]['train']
            val_loader = task_dataloaders[task]['val']

            best_val_metric = -float('inf')
            patience_counter = 0
            patience = 10

            for epoch in range(num_epochs_per_task):

                train_loss = self._train_epoch(
                    train_loader,
                    loss_fn,
                    optimizer,
                    kl_weight,
                    lambda_ewc if task_idx > 0 else 0.0
                )

                val_metric = self._validate(val_loader, task)

                self.training_history[task].append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'val_metric': val_metric
                })

                print(f"Epoch {epoch+1}/{num_epochs_per_task} | "
                      f"Loss: {train_loss:.4f} | Val: {val_metric:.4f}")

                if val_metric > best_val_metric:
                    best_val_metric = val_metric
                    patience_counter = 0
                    self._save_checkpoint(task, epoch, val_metric)
                else:
                    patience_counter += 1

                scheduler.step(val_metric)

                if patience_counter >= patience:
                    print("Early stopping triggered")
                    break


            if task_idx < len(self.tasks) - 1:
                print("\nComputing Fisher Information...")
                self.ewc_regularizer.compute_fisher_information(
                    train_loader,
                    loss_fn,
                    self.device
                )

    # =====================================================
    # TRAIN EPOCH
    # =====================================================
    def _train_epoch(
        self,
        dataloader,
        loss_fn,
        optimizer,
        kl_weight,
        ewc_weight
    ):
        self.model.train()

        total_loss = 0.0

        num_batches = len(dataloader)

        for images, targets in tqdm(dataloader, desc="Training", leave=False):

            images = images.to(self.device)
            targets = targets.to(self.device)

            optimizer.zero_grad()


            predictions = self.model(images, sample=True)


            kl_div = self.model.get_kl_divergence() / num_batches

            loss = loss_fn(predictions, targets, kl_div, kl_weight)


            if ewc_weight > 0:
                loss = loss + ewc_weight * self.ewc_regularizer.get_ewc_loss()

            loss.backward()

            torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)

            optimizer.step()

            total_loss += loss.item()

        return total_loss / num_batches

    # =====================================================
    # VALIDATION
    # =====================================================
    def _validate(self, dataloader, task):
        self.model.eval()

        total_metric = 0.0
        num_batches = 0

        with torch.no_grad():
            for images, targets in dataloader:

                images = images.to(self.device)
                targets = targets.to(self.device)


                predictions = self.model(images, sample=False)

                metric = self._compute_task_metric(predictions, targets, task)

                total_metric += metric
                num_batches += 1

        return total_metric / max(num_batches, 1)

    # =====================================================
    # METRICS
    # =====================================================
    def _compute_task_metric(self, predictions, targets, task):

        if task in ['detection', 'oriented']:
            iou = self._compute_iou(predictions[:, :4], targets[:, :4])
            return iou.mean().item()

        elif task == 'segmentation':
            probs = torch.sigmoid(predictions)

            intersection = (probs * targets).sum(dim=1)
            union = probs.sum(dim=1) + targets.sum(dim=1)

            dice = (2 * intersection + 1e-8) / (union + 1e-8)
            return dice.mean().item()

        elif task == 'pose':
            dist = torch.norm(predictions - targets, dim=1).mean()
            return -dist.item()

        elif task == 'classification':
            preds = predictions.argmax(dim=1)
            targets = targets.argmax(dim=1) if targets.dim() == 2 else targets
            return (preds == targets).float().mean().item()

        return 0.0

    # =====================================================
    # IOU
    # =====================================================
    def _compute_iou(self, pred, target):

        x1 = torch.max(pred[:, 0], target[:, 0])
        y1 = torch.max(pred[:, 1], target[:, 1])
        x2 = torch.min(pred[:, 2], target[:, 2])
        y2 = torch.min(pred[:, 3], target[:, 3])

        inter = (x2 - x1).clamp(min=0) * (y2 - y1).clamp(min=0)

        area_p = (pred[:, 2] - pred[:, 0]) * (pred[:, 3] - pred[:, 1])
        area_t = (target[:, 2] - target[:, 0]) * (target[:, 3] - target[:, 1])

        union = area_p + area_t - inter

        return inter / (union + 1e-8)

    # =====================================================
    # CHECKPOINTING
    # =====================================================
    def _save_checkpoint(self, task, epoch, metric):

        path = self.checkpoint_dir / f"b_yolo_{task}_best.pt"

        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'metric': metric,
            'fisher': self.ewc_regularizer.fisher_information,
            'prev_weights': self.ewc_regularizer.previous_weights
        }, path)

        self.best_metrics[task]['score'] = metric
        self.best_metrics[task]['model_path'] = str(path)

        print(f" Saved best {task} model: {metric:.4f}")

In [4]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset


class DummyBackbone(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(3, 512, 1)
    def forward(self, x):
        return self.conv(x)

# 2. Initialize Model
backbone = DummyBackbone()
model = B_YOLO(backbone, num_classes=10)


def create_dummy_loader(num_samples=10, input_shape=(3, 224, 224), num_classes=10):
    x = torch.randn(num_samples, *input_shape)
    y = torch.randint(0, num_classes, (num_samples,))
    dataset = TensorDataset(x, y)
    return DataLoader(dataset, batch_size=2)

task_dataloaders = {
    'classification': {
        'train': create_dummy_loader(),
        'val': create_dummy_loader()
    }
}


trainer = B_YOLOTrainer(model, tasks=['classification'])


try:
    print("Starting smoke test...")
    trainer.train_sequential_tasks(
        task_dataloaders,
        num_epochs_per_task=1,
        learning_rate=1e-4
    )
    print("\nSuccess! The pipeline runs correctly.")
except Exception as e:
    print(f"\nAn error occurred: {e}")

Starting smoke test...

Task 1/1: CLASSIFICATION


Epoch 1/1 | Loss: 662.3812 | Val: 0.1000
 Saved best classification model: 0.1000

Success! The pipeline runs correctly.
